<a href="https://colab.research.google.com/github/anjelhafidi2003-code/Phonetics-PhonologyFev2026/blob/main/WhisperSubtokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**install + build whisper.cpp (https://github.com/ggml-org/whisper.cpp**)

### Prepare 'Algerian.wav' for transcription

Please ensure your 'Algerian.wav' file is uploaded to `/content/Algerian.wav` before running the next cell. You can upload it by clicking the folder icon on the left sidebar, then the upload icon, and selecting your file.

In [ ]:
import subprocess

WAV_IN  = "/content/Algerian.wav" # Updated for the new audio file
WAV_16K = "/content/audio_16k_Algerian.wav" # New output path for 16k mono WAV

subprocess.run([
    "ffmpeg","-y","-i",WAV_IN,
    "-ar","16000","-ac","1","-c:a","pcm_s16le",
    WAV_16K
], check=True)

### Transcribe 'Algerian.wav' + get tokens

This cell will perform the transcription and save the results as `out_Algerian.json` and `tokens_Algerian.csv` in the `/content/` directory.

In [ ]:
import os, subprocess, json
import re

os.chdir("/content/whisper.cpp")

MODEL = "models/ggml-small.bin"     # Ensure this matches the model downloaded
AUDIO = "/content/audio_16k_Algerian.wav" # Input for whisper-cli is the 16k mono WAV
OUT   = "/content/out_Algerian" # Unique output prefix for Algerian audio

subprocess.run(
    [
        "./build/bin/whisper-cli",
        "-m", MODEL,
        "-f", AUDIO,
        "-l", "auto",        # or "fr", "ar", ...
        "-ojf",
        "-dtw", "base",
        "-of", OUT
    ], check=True)

# Attempt to read the file with utf-8, replacing problematic characters
with open(f"{OUT}.json", "r", encoding="utf-8", errors="replace") as f:
    json_str = f.read()

# Remove invalid control characters that might break JSON parsing.
# This regex matches any control character (0x00-0x1F) except tab (0x09), newline (0x0A), and carriage return (0x0D).
# These specific control characters are generally allowed in JSON strings if escaped.
# Other control characters are not allowed unescaped.
control_chars = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F]')
cleaned_json_str = control_chars.sub(lambda x: f'\\u{ord(x.group()):04x}', json_str)

# Now try parsing the cleaned string
j = json.loads(cleaned_json_str)

rows = []
for seg in j.get("transcription", []):
    for tok in seg.get("tokens", []):
        offs = tok.get("offsets", {})  # ms, when available
        rows.append({
            "token": tok.get("text"),
            "start_ms": offs.get("from"),
            "end_ms": offs.get("to"),
            "p": tok.get("p"),
            "id": tok.get("id"),
        })

__import__("pandas").DataFrame(rows).to_csv(f"{OUT}_tokens.csv", index=False)

# print first tokens
print("\nFirst 40 tokens:")
for r in rows[:40]:
    print(r)

# print ALL tokens
print(f"\nTotal tokens: {len(rows)}\n")
print("All tokens:")
for i, r in enumerate(rows, 1):
    print(f"{i}. {r}")

You can now manually download the `out_Algerian.json` and `out_Algerian_tokens.csv` files. Click on the folder icon on the left sidebar, navigate to `/content/`, right-click on the desired file, and select 'Download'.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, subprocess, textwrap, sys

def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

# system deps
run(["apt-get","-qq","update"])
run(["apt-get","-qq","install","-y","ffmpeg","cmake","build-essential","git"])

# clone + build
os.chdir("/content")
if not os.path.exists("whisper.cpp"):
    run(["git","clone","https://github.com/ggml-org/whisper.cpp"])
os.chdir("/content/whisper.cpp")

run(["cmake","-B","build"])
run(["cmake","--build","build","-j"])


 **download a multilingual model (could be base, small, medium, large)
!Attention: large model may crash Google Colab**

In [ ]:
import os, subprocess

def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


os.chdir("/content/whisper.cpp")
run(["bash","./models/download-ggml-model.sh","small"])  # You can change the model


**ensure WAV format (16k mono)**

In [ ]:
import subprocess

WAV_IN  = "/content/arabic.wav"         #change the file path
WAV_16K = "/content/audio_16k.wav"

subprocess.run([
    "ffmpeg","-y","-i",WAV_IN,
    "-ar","16000","-ac","1","-c:a","pcm_s16le",
    WAV_16K
], check=True)

**Transcribe + get tokens + (experimental) token timestamps + probs**

In [ ]:
import os, subprocess, json
import re

os.chdir("/content/whisper.cpp")

MODEL = "models/ggml-small.bin"     #Be sure the model matches the one you selected in the previous cell.
AUDIO = "/content/audio_16k.wav"
OUT   = "/content/out"

subprocess.run(
    [
        "./build/bin/whisper-cli",
        "-m", MODEL,
        "-f", AUDIO,
        "-l", "auto",        # or "fr", "ar", ...
        "-ojf",
        "-dtw", "base",
        "-of", OUT
    ], check=True)

# Attempt to read the file with utf-8, replacing problematic characters
with open("/content/out.json", "r", encoding="utf-8", errors="replace") as f:
    json_str = f.read()

# Remove invalid control characters that might break JSON parsing.
# This regex matches any control character (0x00-0x1F) except tab (0x09), newline (0x0A), and carriage return (0x0D).
# These specific control characters are generally allowed in JSON strings if escaped.
# Other control characters are not allowed unescaped.
control_chars = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F]')
cleaned_json_str = control_chars.sub(lambda x: f'\\u{ord(x.group()):04x}', json_str)

# Now try parsing the cleaned string
j = json.loads(cleaned_json_str)

rows = []
for seg in j.get("transcription", []):
    for tok in seg.get("tokens", []):
        offs = tok.get("offsets", {})  # ms, when available
        rows.append({
            "token": tok.get("text"),
            "start_ms": offs.get("from"),
            "end_ms": offs.get("to"),
            "p": tok.get("p"),
            "id": tok.get("id"),
        })

__import__("pandas").DataFrame(rows).to_csv("/content/tokens.csv", index=False)

# print first tokens
for r in rows[:40]:
    print(r)

# print ALL tokens
print(f"Total tokens: {len(rows)}\n")
for i, r in enumerate(rows, 1):
    print(f"{i}. {r}")

You can also manually download the `tokens.csv` file. Click on the folder icon on the left sidebar, navigate to `/content/tokens.csv`, right-click on the file, and select 'Download'.